## NA2M - full pipeline: arm A vs arm B (GAMI-Net) vs arm C (NA2M)

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

SEED = 42
set_seed(SEED)

### Data

In [ ]:
from na2m.data.data_utils import load_compas, preprocess

df = load_compas(r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\datasets\raw\compas-scores-two-years.csv')
X, y, feature_meta = preprocess(df)

print(f'Samples: {X.shape[0]}, Features: {X.shape[1]}')
for fm in feature_meta:
    print(f'  {fm.name:25s} type={fm.type}')

### Tune NA2M hyperparameters (arm B — used for both B and C)

In [ ]:
import importlib.util
from pathlib import Path
from na2m.utils.config import load_na2m_search_config
from na2m.data.data_utils import split

_SEARCH_CONFIG = r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas_na2m_search.yaml'

# Load tune_fold without touching the na2m package namespace
_spec = importlib.util.spec_from_file_location(
    "tune_na2m_script",
    Path(_SEARCH_CONFIG).parent.parent / "scripts" / "na2m" / "tune_na2m.py",
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
tune_fold = _mod.tune_fold

fixed_params, search_space = load_na2m_search_config(_SEARCH_CONFIG)

# Single shared split for both the tuner and the arm training cells below
X_train, X_val, X_test, y_train, y_val, y_test = split(
    X, y, fixed_params['val_frac'], fixed_params['test_frac'], seed=SEED
)
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

TUNED_PATH = Path(_SEARCH_CONFIG).parent / 'compas-scores-two-years_na2m_interactions_tuned.yaml'

tune_fold(
    fixed_params, search_space, feature_meta,
    X_train, y_train, X_val, y_val,
    TUNED_PATH,
    with_interactions=True,
)

### Config

In [ ]:
from na2m.utils.config import load_na2m_config

config = load_na2m_config(str(TUNED_PATH))
config.top_m = 20
config.eta_prune = 0.01

print(config)

### Loaders

In [ ]:
from nam.data.dataset import NAMDataset
from torch.utils.data import DataLoader

train_dataset = NAMDataset(X_train, y_train)
val_dataset   = NAMDataset(X_val, y_val)
pool_dataset  = NAMDataset(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
test_dataset  = NAMDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=config.batch_size, shuffle=False)
pool_loader  = DataLoader(pool_dataset,  batch_size=config.batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=config.batch_size, shuffle=False)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Pool: {len(pool_dataset)}, Test: {len(test_dataset)}')

### Helpers

In [ ]:
from na2m.models.na2m import NA2M
from na2m.training.fit_na2m import fit_na2m
from nam.training.metrics import auroc

def build_model():
    return NA2M(
        num_features=X.shape[1], feature_meta=feature_meta,
        num_units=config.num_units, hidden_sizes=config.hidden_sizes,
        dropout=config.dropout, feature_dropout=config.feature_dropout,
        activation=config.activation, inter_units=config.inter_units,
        inter_hidden=config.inter_hidden,
    )

def test_auroc(model):
    model.eval()
    logits_all, targets_all = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in test_loader:
            logits, _ = model(X_batch)
            logits_all.append(logits)
            targets_all.append(y_batch)
    return auroc(torch.cat(logits_all), torch.cat(targets_all))

def print_result(label, auc, result):
    pairs = result['active_pairs']
    print(f'{label}  AUROC={auc:.4f}  interactions={len(pairs)}')
    for j, k in pairs:
        print(f'  {feature_meta[j].name} x {feature_meta[k].name}')

### Arm A - mains only

In [ ]:
set_seed(SEED)
model_a = build_model()

result_a = fit_na2m(
    model_a, train_loader, val_loader, pool_loader, config,
    with_interactions=False,
    with_concurvity_filter=False,
)

auc_a = test_auroc(model_a)
print_result('Arm A (mains only)', auc_a, result_a)

### Arm B - GAMI-Net (interactions, no concurvity filter)

In [ ]:
set_seed(SEED)
model_b = build_model()

result_b = fit_na2m(
    model_b, train_loader, val_loader, pool_loader, config,
    with_interactions=True,
    with_concurvity_filter=False,
)

auc_b = test_auroc(model_b)
print_result('Arm B (GAMI-Net, no filter)', auc_b, result_b)

### Arm C - NA2M (interactions + concurvity filter)

In [ ]:
set_seed(SEED)
model_c = build_model()

result_c = fit_na2m(
    model_c, train_loader, val_loader, pool_loader, config,
    with_interactions=True,
    with_concurvity_filter=True,
)

auc_c = test_auroc(model_c)
print_result('Arm C (NA2M, with filter)', auc_c, result_c)

### Comparison

In [ ]:
print(f'Arm A (mains only):          AUROC={auc_a:.4f}  interactions=0')
print(f'Arm B (GAMI-Net, no filter): AUROC={auc_b:.4f}  interactions={len(result_b["active_pairs"])}')
print(f'Arm C (NA2M, with filter):   AUROC={auc_c:.4f}  interactions={len(result_c["active_pairs"])}')